In [2]:
import sympy as sp

x = sp.symbols('x')

with sp.evaluate(False):
    expr = sp.Integer(1) + sp.Integer(1) + sp.Integer(1)

q = sp.Integer(1)
query = sp.core.basic._make_find_query(q)
results = list(filter(query, sp.core.basic._preorder_traversal(expr)))
print(list(sp.core.basic._preorder_traversal(expr)))
print(results)

[1 + (1 + 1), 1 + 1, 1, 1, 1]
[1, 1, 1]


In [3]:
from sympy import sin, cos, tan, Wild, Mul, Integer, Add
p = Wild("p")
q = Wild("q")

rules_without_consts = {
    "TR2a": (tan(p), (sin(p))/(cos(p))),
    "TR2ainv": ((sin(p))/(cos(p)), tan(p)),
    "TR5": (sin(p)**2, 1 - cos(p)**2),
    "TR5inv": (1 - cos(p)**2, sin(p)**2),
    "TR6": (cos(p)**2, 1 - sin(p)**2),
    "TR6inv": (1 - sin(p)**2, cos(p)**2),
    "TR7": (cos(p)**2, (1 + cos(2*p))/2),
    "TR8a": (sin(p)*cos(q), (1/2) * (sin(p + q) + sin(p - q))),
    # "TR8ainv": (sin(p) + sin(q), 2 * sin((p + q)/2) * cos((p - q)/2)),
    "TR8b": (cos(p) * sin(q), (1/2) * (sin(p + q) - sin(p - q))),
    # "TR8binv": (sin(p) - sin(q), 2 * cos((p + q) / 2) * sin((p - q)/2)),
    "TR8c": (cos(p) * cos(q), (1/2) * (cos(p + q) + cos(p - q))),
    # "TR8cinv": (cos(p) + cos(q), 2 * cos((p + q) / 2) * cos((p - q)/2)),
    "TR8d": (sin(p) * sin(q), - (1/2) * (cos(p + q) - cos(p - q))),
    
    "TR9a": (sin(p) + sin(q), 2 * sin((p + q)/2) * cos((p - q)/2)),
    "TR9b": (sin(p) - sin(q), 2 * cos((p + q)/2) * sin((p - q)/2)),
    "TR9c": (cos(p) + cos(q), 2 * cos((p + q)/2) * cos((p - q)/2)),
    "TR9d": (cos(p) - cos(q), -2 * sin((p + q)/2) * sin((p - q)/2)),
    "TR10a": (sin(p + q), sin(p) * cos(q) + cos(p) * sin(q)),
    "TR10b": (sin(p - q), sin(p) * cos(q) - cos(p) * sin(q)),
    "TR10c": (cos(p + q), cos(p) * cos(q) - sin(p) * sin(q)),
    "TR10d": (cos(p - q), cos(p) * cos(q) + sin(p) * sin(q)),
    "TR11a": (sin(p), 2 * sin(p/2) * cos(p/2)),
    "TR11b": (cos(2*p), cos(p)**2 - sin(p)**2),
    "TR11c": (cos(2*p), 2 * cos(p)**2 - 1),
    "TR11d": (cos(2*p), 1 - 2 * sin(p)**2),
    "TR12a": (tan(p + q), (tan(p) + tan(q))/(1 - tan(p) * tan(q))),
    "TR12b": (tan(p - q), (tan(p) - tan(q))/(1 + tan(p) * tan(q))),
    "mulbyone": (p, Mul(Integer(1), p, evaluate=False)),
    "complexifyOneA": (Integer(1), Add(sin(x)**2, cos(x)**2, evaluate=False)),

}


In [4]:
def _replace_nth_suboccurence(expr, n, pattern, replacement_pattern, counter):
  try:
    match_dict = expr.match(pattern)
  except Exception:
    match_dict = None
  if match_dict != None: # if no variables are bound, expr.match can return {}, which is still a match. but python interprets this as false
    counter[0] += 1
    if counter[0] == n:
      with sp.evaluate(False):
        return replacement_pattern.subs(match_dict)
  if expr.args:
    with sp.evaluate(False):
      return expr.func(*[
          _replace_nth_suboccurence(arg, n, pattern, replacement_pattern, counter)
          for arg in expr.args
      ])
  else:
    return expr

def replace_nth_suboccurence(expr, n, pattern, replacement_pattern):
  counter = [0]

  return _replace_nth_suboccurence(expr, n, pattern, replacement_pattern, counter)



In [5]:
import random

def random_replacement(expr, pattern, replacement_pattern):
  try:
    num_matches = expr.count(pattern)
  except Exception:
    # SymPy bugs with complex expressions - skip this replacement
    return expr
  if num_matches == 0:
    return expr
  if num_matches == 1:
    n = 1
  else:
    n = random.randint(1, num_matches)
  return replace_nth_suboccurence(expr, n, pattern, replacement_pattern)

In [6]:
import threading
import time

def timeout(seconds):
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = [None]
            exception = [None]
            
            def target():
                try:
                    result[0] = func(*args, **kwargs)
                except Exception as e:
                    exception[0] = e
            
            thread = threading.Thread(target=target)
            thread.daemon = True
            thread.start()
            thread.join(timeout=seconds)
            
            if thread.is_alive():
                print(f"Function timed out after {seconds} seconds")
                return None
            
            if exception[0]:
                raise exception[0]
            
            return result[0]
        
        return wrapper
    return decorator

In [7]:
@timeout(5)
def repeatedly_apply_random_replacement(expr, rules, num_times):
  # use j to ensure that function stops even if no more replacements can be applied
  result = expr
  i = 0
  j = 0
  while (i < num_times) and (j < num_times * 10):
    j += 1
    rule = random.choice(list(rules.values()))
    pattern, replacement_pattern = rule
    print(f"Rule: {pattern} -> {replacement_pattern} on {result}")
    try:
      new_result = random_replacement(result, pattern, replacement_pattern)
    except:
      print("Something went wrong")
    print(f"Now it is {new_result}, i = {i}, j = {j}")
    if new_result != result:
      result = new_result
      i += 1
  return result

In [7]:
expr6 = sin(x)**2
print(expr6)
complicated_expr = repeatedly_apply_random_replacement(expr6, rules_without_consts, 15)

sin(x)**2
Rule: sin(p_) - sin(q_) -> 2*sin(p_/2 - q_/2)*cos(p_/2 + q_/2) on sin(x)**2
Now it is sin(x)**2, i = 0, j = 1
Rule: sin(p_)*sin(q_) -> -0.5*cos(p_ + q_) + 0.5*cos(-p_ + q_) on sin(x)**2
Now it is 0.5*cos(-x + x) - 0.5*cos(x + x), i = 0, j = 2
Rule: tan(p_) -> sin(p_)/cos(p_) on 0.5*cos(-x + x) - 0.5*cos(x + x)
Now it is 0.5*cos(-x + x) - 0.5*cos(x + x), i = 1, j = 3
Rule: cos(p_)*cos(q_) -> 0.5*cos(p_ + q_) + 0.5*cos(-p_ + q_) on 0.5*cos(-x + x) - 0.5*cos(x + x)
Now it is 0.5*cos(-x + x) - 0.5*cos(x + x), i = 1, j = 4
Rule: cos(2*p_) -> -sin(p_)**2 + cos(p_)**2 on 0.5*cos(-x + x) - 0.5*cos(x + x)
Now it is 0.5*(-sin((-x + x)/2)**2 + cos((-x + x)/2)**2) - 0.5*cos(x + x), i = 1, j = 5
Rule: -sin(-p_ + q_) -> sin(p_)*cos(q_) - sin(q_)*cos(p_) on 0.5*(-sin((-x + x)/2)**2 + cos((-x + x)/2)**2) - 0.5*cos(x + x)
Now it is 0.5*(-sin((-x + x)/2)**2 + cos((-x + x)/2)**2) - 0.5*cos(x + x), i = 2, j = 6
Rule: cos(p_)**2 -> cos(2*p_)/2 + 1/2 on 0.5*(-sin((-x + x)/2)**2 + cos((-x + x)/2)**

In [ ]:
i = 0
count = 0
while i < 50:
    print(f"Generating expression {i+1}/50...")
    try:
        start_expr = sin(x)**2
        result = repeatedly_apply_random_replacement(start_expr, rules_without_consts, 15)
        result = sp.expand(result)
        
        if result is not None:
            with open('data.txt', 'a') as f:
                f.write(str(result) + '\n')
            count += 1
            i += 1
            print(f"  Success! Generated and wrote expression {i}")
        else:
            print(f"  Got None, retrying...")
    except Exception as e:
        print(f"  Error occurred: {e}, retrying...")

print(f"\nSuccessfully wrote {count} expressions to data.txt")

Generating expression 1/50...
Rule: tan(p_ + q_) -> (tan(p_) + tan(q_))/(-tan(p_)*tan(q_) + 1) on sin(x)**2
Now it is sin(x)**2, i = 0, j = 1
Rule: sin(p_) - sin(q_) -> 2*sin(p_/2 - q_/2)*cos(p_/2 + q_/2) on sin(x)**2
Now it is sin(x)**2, i = 0, j = 2
Rule: cos(2*p_) -> -sin(p_)**2 + cos(p_)**2 on sin(x)**2
Now it is sin(x)**2, i = 0, j = 3
Rule: -tan(-p_ + q_) -> (tan(p_) - tan(q_))/(tan(p_)*tan(q_) + 1) on sin(x)**2
Now it is sin(x)**2, i = 0, j = 4
Rule: sin(p_) - sin(q_) -> 2*sin(p_/2 - q_/2)*cos(p_/2 + q_/2) on sin(x)**2
Now it is sin(x)**2, i = 0, j = 5
Rule: sin(p_)*cos(q_) -> 0.5*sin(p_ + q_) - 0.5*sin(-p_ + q_) on sin(x)**2
Now it is sin(x)**2, i = 0, j = 6
Rule: cos(p_) + cos(q_) -> 2*cos(p_/2 - q_/2)*cos(p_/2 + q_/2) on sin(x)**2
Now it is sin(x)**2, i = 0, j = 7
Rule: p_ -> 1*p_ on sin(x)**2
Now it is sin(x)**(1*2), i = 0, j = 8
Rule: sin(p_) -> 2*sin(p_/2)*cos(p_/2) on sin(x)**(1*2)
Now it is (2*sin(x/2)*cos(x/2))**(1*2), i = 1, j = 9
Rule: cos(2*p_) -> 2*cos(p_)**2 - 1 on

In [14]:
sp.trigsimp(sin(x)**4*cos(x)**2 + sin(x)**4 + 2*sin(x)**2*cos(x)**4 + sin(x)**2*cos(x)**2 - sin(x)**2 + cos(x)**6 - 2*cos(x)**2 + 1
, method='fu')

sin(x)**2

In [16]:
from sympy import pi
sin(pi)

0